# SAINT

Student-blocked nested evaluation of SAINT on two binary prediction tasks:

- **First attempt:** `firstattemptcorrect`
- **After feedback:** `eventualcorrect`

Run the setup cell, then either task cell. Each task uses its own fixed outer-fold file and output directory. Hyperparameters and the final training epoch count are selected exclusively from inner validation folds.


In [ ]:
from pathlib import Path
import pandas as pd

# Point this to the interaction-level FeedBook export.
DATA_PATH = Path("data/feedbook_interactions.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. Update DATA_PATH before running."
    )

logging_data = pd.read_csv(DATA_PATH)
print(f"Loaded {len(logging_data):,} interactions from {DATA_PATH}")


## First-attempt prediction


In [ ]:
# SAINT First Attempt

import os
import random
import itertools
from statistics import mean, stdev

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==== IMPORT SAINT FROM PYKT ====
from pykt.models import saint

SAINT = saint.SAINT


# ============================================================
# REPRODUCIBILITY AND DEVICE
# ============================================================

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")


# ============================================================
# SETTINGS
# ============================================================

seq_len = 20
batch_size = 8
n_splits = 3
max_epochs = 100
patience = 10

model_name = "SAINT"
kc_name = "actionableelementid"
question_col = "KC (actionableelementid)"
correct_col = "firstattemptcorrect"
student_col = "Anon Student Id"

fixed_fold_file = "data/fixed_outer_folds.csv"
oof_output_dir = "SAINT_FA"

os.makedirs(oof_output_dir, exist_ok=True)


# ============================================================
# LOAD DATA
# ============================================================

# Load your dataframe before running this script, for example:
# logging_data = pd.read_csv("your_file.csv")

if "logging_data" not in globals():
    raise NameError(
        "logging_data is not defined. Load your dataframe into a variable "
        "named logging_data before running this script."
    )

logging_data = logging_data.copy().reset_index(drop=True)

# row_id must be created on the full original dataframe before model-specific
# filtering so OOF predictions can align across all DLKT models.
if "row_id" not in logging_data.columns:
    logging_data["row_id"] = np.arange(len(logging_data), dtype=np.int64)

if logging_data["row_id"].duplicated().any():
    raise ValueError("row_id must be unique in the full logging_data dataframe.")

required_source_columns = {
    "row_id",
    question_col,
    correct_col,
    student_col,
}

missing_source_columns = required_source_columns - set(logging_data.columns)

if missing_source_columns:
    raise ValueError(
        f"logging_data is missing required columns: "
        f"{sorted(missing_source_columns)}"
    )


# ============================================================
# FIXED OUTER FOLDS
# ============================================================

# Remove any old fold columns left by notebook reruns before merging.
old_fold_columns = [
    column
    for column in logging_data.columns
    if column == "outer_fold" or column.startswith("outer_fold_")
]

if old_fold_columns:
    logging_data = logging_data.drop(columns=old_fold_columns)

if os.path.exists(fixed_fold_file):
    fixed_folds = pd.read_csv(fixed_fold_file)

    required_fold_columns = {"row_id", "outer_fold"}
    missing_fold_columns = required_fold_columns - set(fixed_folds.columns)

    if missing_fold_columns:
        raise ValueError(
            f"{fixed_fold_file} is missing columns: "
            f"{sorted(missing_fold_columns)}"
        )

    if fixed_folds["row_id"].duplicated().any():
        raise ValueError(
            f"{fixed_fold_file} contains duplicate row_id values."
        )

    found_folds = sorted(
        fixed_folds["outer_fold"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    expected_folds = list(range(1, n_splits + 1))

    if found_folds != expected_folds:
        raise ValueError(
            f"{fixed_fold_file} contains folds {found_folds}, "
            f"but this run expects {expected_folds}."
        )

    logging_data = logging_data.merge(
        fixed_folds[["row_id", "outer_fold"]],
        on="row_id",
        how="left",
        validate="one_to_one",
    )

    missing_valid_fold = (
        logging_data[student_col].notna()
        & logging_data["outer_fold"].isna()
    )

    if missing_valid_fold.any():
        raise ValueError(
            "Some rows with valid student IDs are missing outer-fold "
            "assignments. The fixed-fold file may belong to another "
            "dataset or row ordering."
        )

    print(f"Loaded fixed folds from {fixed_fold_file}")

else:
    base_for_folds = logging_data.dropna(
        subset=[student_col]
    ).copy()

    if base_for_folds[student_col].nunique() < n_splits:
        raise ValueError(
            f"At least {n_splits} unique students are required to create "
            f"{n_splits} grouped outer folds."
        )

    base_for_folds["outer_fold"] = -1

    outer_gkf = GroupKFold(n_splits=n_splits)

    for fold_idx, (_, test_idx) in enumerate(
        outer_gkf.split(
            base_for_folds,
            groups=base_for_folds[student_col],
        ),
        start=1,
    ):
        base_for_folds.iloc[
            test_idx,
            base_for_folds.columns.get_loc("outer_fold"),
        ] = fold_idx

    fixed_folds = base_for_folds[
        ["row_id", "outer_fold"]
    ].copy()

    fixed_folds.to_csv(
        fixed_fold_file,
        index=False,
    )

    logging_data = logging_data.merge(
        fixed_folds,
        on="row_id",
        how="left",
        validate="one_to_one",
    )

    print(f"Saved fixed folds to {fixed_fold_file}")

logging_data["outer_fold"] = logging_data["outer_fold"].astype("Int64")


# ============================================================
# MODEL-SPECIFIC PREPROCESSING
# ============================================================

logging_model = logging_data.dropna(
    subset=[
        question_col,
        correct_col,
        student_col,
        "outer_fold",
    ]
).copy()

logging_model[correct_col] = logging_model[correct_col].astype(int)
logging_model["outer_fold"] = logging_model["outer_fold"].astype(int)

invalid_labels = ~logging_model[correct_col].isin([0, 1])

if invalid_labels.any():
    bad_values = sorted(
        logging_model.loc[invalid_labels, correct_col]
        .unique()
        .tolist()
    )

    raise ValueError(
        f"{correct_col} must be binary 0/1. Found: {bad_values}"
    )

# Map each original row to its student so student_id is saved with OOF rows.
student_id_lookup = (
    logging_model[["row_id", student_col]]
    .drop_duplicates(subset=["row_id"])
    .set_index("row_id")[student_col]
    .to_dict()
)

print(f"seq_len: {seq_len}")
print(f"Usable rows: {len(logging_model)}")
print(f"Unique students: {logging_model[student_col].nunique()}")

# Global KC mapping for this model run.
# 0 is reserved exclusively for padding.
all_qids = logging_model[question_col].dropna().unique()

qid_to_index = {
    qid: index + 1
    for index, qid in enumerate(all_qids)
}

num_questions = len(qid_to_index) + 1

print(f"Number of KC IDs including padding: {num_questions}")


# ============================================================
# DATASET
# ============================================================

class KTDataFromLogging(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        seq_len: int,
        question_col: str,
        correct_col: str,
        student_col: str,
        qid_to_index: dict,
    ) -> None:
        self.seq_len = seq_len
        self.samples = []

        df = df.copy()

        df["qid_index"] = (
            df[question_col]
            .map(qid_to_index)
            .fillna(0)
            .astype(int)
        )

        # Preserve the existing dataframe order within each student,
        # exactly like the SKVMN wrapper.
        for _, group in df.groupby(student_col, sort=False):
            q_seq = group["qid_index"].tolist()
            r_seq = group[correct_col].astype(int).tolist()
            row_seq = group["row_id"].astype(int).tolist()

            for start in range(0, len(q_seq), seq_len - 1):
                end = min(start + seq_len, len(q_seq))

                if end - start < 2:
                    break

                q_chunk = q_seq[start:end]
                r_chunk = r_seq[start:end]
                row_chunk = row_seq[start:end]

                pad_len = seq_len - len(q_chunk)

                if pad_len > 0:
                    q_chunk += [0] * pad_len
                    r_chunk += [0] * pad_len
                    row_chunk += [-1] * pad_len

                self.samples.append(
                    (
                        torch.tensor(q_chunk, dtype=torch.long),
                        torch.tensor(r_chunk, dtype=torch.long),
                        torch.tensor(row_chunk, dtype=torch.long),
                    )
                )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        return self.samples[index]


# ============================================================
# SAINT HYPERPARAMETER GRID
# ============================================================

# Same exhaustive search space as the original SAINT wrapper,
# searched with ordinary Python exhaustive loops.
param_search_space = {
    "d_model": [64, 128, 256],
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "num_attn_heads": [4, 8],
    "dropout": [0.1, 0.3, 0.5],
    "n_blocks": [1, 2, 4],
}


# ============================================================
# HELPERS
# ============================================================

def rmse_score(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return float(
        np.sqrt(
            np.mean((y_true - y_pred) ** 2)
        )
    )


def make_saint_model(params: dict) -> nn.Module:
    """
    This experiment varies ONE KC representation at a time.

    SAINT normally supports a question/exercise stream (num_q) and a
    concept/category stream (num_c). Here the selected KC column is used
    only as the concept/category stream so it is not embedded twice.
    """
    return SAINT(
        num_q=0,
        num_c=num_questions,
        seq_len=seq_len,
        emb_size=params["d_model"],
        num_attn_heads=params["num_attn_heads"],
        dropout=params["dropout"],
        n_blocks=params["n_blocks"],
        emb_type="qid",
    ).to(device)


def saint_forward(
    model: nn.Module,
    q_batch: torch.Tensor,
    r_batch: torch.Tensor,
) -> torch.Tensor:
    """
    pyKT SAINT alignment for our full-sequence wrapper.

    Encoder:
        full KC sequence, length L

    Decoder:
        responses r[:, :-1], length L-1
        SAINT prepends its own start token -> decoder length L

    Therefore:
        output[:, 1:] predicts r[:, 1:]
    """
    dummy_exercise = torch.zeros_like(q_batch)

    outputs = model(
        in_ex=dummy_exercise,
        in_cat=q_batch,
        in_res=r_batch[:, :-1],
    )

    if outputs.shape != q_batch.shape:
        raise RuntimeError(
            f"Unexpected SAINT output shape {tuple(outputs.shape)}; "
            f"expected {tuple(q_batch.shape)}."
        )

    return outputs


def saint_loss(
    model: nn.Module,
    q_batch: torch.Tensor,
    r_batch: torch.Tensor,
    criterion: nn.Module,
):
    """
    SAINT-specific training alignment.

    output[:, 1:] -> r_batch[:, 1:]
    Position 0 and padding are excluded.
    """
    outputs = saint_forward(
        model,
        q_batch,
        r_batch,
    )

    valid_mask = q_batch[:, 1:] != 0

    if not valid_mask.any():
        return None

    loss = criterion(
        outputs[:, 1:][valid_mask],
        r_batch[:, 1:].float()[valid_mask],
    )

    return loss


def collect_saint_predictions(
    model: nn.Module,
    loader: DataLoader,
    include_row_ids: bool = False,
):
    """
    SAINT evaluation/OOF alignment:

        outputs[b, t] -> r_batch[b, t] -> rowid_batch[b, t]

    Position 0 is excluded.
    Padding is excluded because KC ID 0 is reserved for padding.
    """
    model.eval()

    preds_all = []
    labels_all = []
    row_ids_all = []

    with torch.no_grad():
        for q_batch, r_batch, rowid_batch in loader:
            q_batch = q_batch.to(device)
            r_batch = r_batch.to(device)

            outputs = saint_forward(
                model,
                q_batch,
                r_batch,
            )

            for batch_index in range(q_batch.size(0)):
                for time_index in range(1, seq_len):
                    if q_batch[batch_index, time_index].item() == 0:
                        continue

                    pred = outputs[
                        batch_index,
                        time_index,
                    ].item()

                    label = r_batch[
                        batch_index,
                        time_index,
                    ].item()

                    if include_row_ids:
                        row_id = rowid_batch[
                            batch_index,
                            time_index,
                        ].item()

                        if row_id == -1:
                            continue

                        row_ids_all.append(int(row_id))

                    preds_all.append(float(pred))
                    labels_all.append(int(label))

    if include_row_ids:
        return preds_all, labels_all, row_ids_all

    return preds_all, labels_all


def calculate_metrics(labels, predictions) -> dict:
    labels = np.asarray(labels, dtype=int)
    predictions = np.asarray(predictions, dtype=float)
    binary_predictions = (predictions > 0.5).astype(int)

    auc = (
        roc_auc_score(labels, predictions)
        if len(np.unique(labels)) > 1
        else np.nan
    )

    return {
        "auc": auc,
        "accuracy": accuracy_score(labels, binary_predictions),
        "rmse": rmse_score(labels, predictions),
        "mae": mean_absolute_error(labels, predictions),
        "precision": precision_score(
            labels,
            binary_predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            labels,
            binary_predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            labels,
            binary_predictions,
            zero_division=0,
        ),
    }


def mean_std(values):
    valid_values = [
        float(value)
        for value in values
        if not pd.isna(value)
    ]

    if not valid_values:
        return np.nan, np.nan

    return (
        mean(valid_values),
        stdev(valid_values)
        if len(valid_values) > 1
        else 0.0,
    )


def parameter_combinations():
    keys = list(param_search_space.keys())

    for values in itertools.product(
        *(param_search_space[key] for key in keys)
    ):
        yield dict(zip(keys, values))


# ============================================================
# STORAGE
# ============================================================

all_preds_all_folds = []
all_labels_all_folds = []
all_rowids_all_folds = []
all_foldnums_all_folds = []

auc_per_fold = []
acc_per_fold = []
rmse_per_fold = []
mae_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []


# ============================================================
# OUTER CROSS-VALIDATION
# ============================================================

for fold in range(1, n_splits + 1):
    print(f"\n=== Outer Fold {fold}/{n_splits} ===")

    train_val_df = logging_model[
        logging_model["outer_fold"] != fold
    ].copy()

    test_df = logging_model[
        logging_model["outer_fold"] == fold
    ].copy()

    print(f"Train/validation rows: {len(train_val_df)}")
    print(f"Test rows: {len(test_df)}")
    print(
        f"Train/validation students: "
        f"{train_val_df[student_col].nunique()}"
    )
    print(f"Test students: {test_df[student_col].nunique()}")

    if train_val_df[student_col].nunique() < n_splits:
        raise ValueError(
            f"Outer fold {fold} does not contain enough training students "
            f"for {n_splits}-fold inner GroupKFold."
        )

    inner_cv = GroupKFold(n_splits=n_splits)
    inner_groups = train_val_df[student_col]

    # --------------------------------------------------------
    # INNER-CV HYPERPARAMETER SEARCH
    # --------------------------------------------------------

    best_params = None
    best_mean_inner_auc = -np.inf
    recommended_epochs = 1

    combinations = list(parameter_combinations())

    # Match the same compact trial-progress display used by the other runs.
    progress_bar = tqdm(total=len(combinations))

    best_trial_index = None

    for combination_index, params in enumerate(combinations, start=1):
        auc_scores = []
        best_epochs = []

        for inner_train_idx, inner_val_idx in inner_cv.split(
            train_val_df,
            groups=inner_groups,
        ):
            inner_train_df = train_val_df.iloc[
                inner_train_idx
            ].copy()

            inner_val_df = train_val_df.iloc[
                inner_val_idx
            ].copy()

            train_dataset = KTDataFromLogging(
                inner_train_df,
                seq_len,
                question_col,
                correct_col,
                student_col,
                qid_to_index,
            )

            val_dataset = KTDataFromLogging(
                inner_val_df,
                seq_len,
                question_col,
                correct_col,
                student_col,
                qid_to_index,
            )

            train_loader = DataLoader(
                train_dataset,
                batch_size=batch_size,
                shuffle=True,
            )

            val_loader = DataLoader(
                val_dataset,
                batch_size=batch_size,
                shuffle=False,
            )

            model = make_saint_model(params)
            criterion = nn.BCELoss()

            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=params["learning_rate"],
            )

            best_auc_inner = -np.inf
            best_epoch_inner = 1
            no_improve = 0

            for epoch in range(max_epochs):
                model.train()

                for q_batch, r_batch, _ in train_loader:
                    q_batch = q_batch.to(device)
                    r_batch = r_batch.to(device)

                    optimizer.zero_grad()

                    loss = saint_loss(
                        model,
                        q_batch,
                        r_batch,
                        criterion,
                    )

                    if loss is None:
                        continue

                    loss.backward()
                    optimizer.step()

                val_preds, val_labels = collect_saint_predictions(
                    model,
                    val_loader,
                    include_row_ids=False,
                )

                if (
                    len(val_labels) > 0
                    and len(np.unique(val_labels)) > 1
                ):
                    val_auc = roc_auc_score(
                        val_labels,
                        val_preds,
                    )

                    # Same rule as the SKVMN wrapper:
                    # any strictly larger AUC counts as improvement.
                    if val_auc > best_auc_inner:
                        best_auc_inner = val_auc
                        best_epoch_inner = epoch + 1
                        no_improve = 0
                    else:
                        no_improve += 1

                    if no_improve >= patience:
                        break

            auc_scores.append(best_auc_inner)
            best_epochs.append(best_epoch_inner)

        mean_inner_auc = (
            float(np.mean(auc_scores))
            if auc_scores
            else 0.0
        )

        print(
            f"Mean inner AUC for {params}: "
            f"{mean_inner_auc:.6f}"
        )

        if mean_inner_auc > best_mean_inner_auc:
            best_mean_inner_auc = mean_inner_auc
            best_params = params.copy()
            recommended_epochs = max(
                1, int(np.median(best_epochs))
            )
            # Trial numbering starts at 0 to match the other model runs.
            best_trial_index = combination_index - 1

        # Update the description first, then increment the completed-trial count.
        if best_trial_index is not None:
            progress_bar.set_description(
                f"Best trial: {best_trial_index}. "
                f"Best value: {best_mean_inner_auc:.6g}"
            )

        progress_bar.update(1)

    progress_bar.close()

    if best_params is None:
        raise RuntimeError(
            f"No valid SAINT parameter configuration was selected "
            f"for outer fold {fold}."
        )

    print(
        f"\nBest params for fold {fold}: "
        f"{best_params}, AUC: {best_mean_inner_auc:.6f}"
    )


    # --------------------------------------------------------
    # RETRAIN BEST CONFIGURATION ON TRAIN_VAL_DF
    # --------------------------------------------------------

    train_dataset = KTDataFromLogging(
        train_val_df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index,
    )

    test_dataset = KTDataFromLogging(
        test_df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    best_model = make_saint_model(best_params)
    criterion = nn.BCELoss()

    optimizer = torch.optim.Adam(
        best_model.parameters(),
        lr=best_params["learning_rate"],
    )

    # Retrain on every outer-training student for the epoch count
    # selected from the inner validation folds.
    for epoch in range(recommended_epochs):
        best_model.train()

        for q_batch, r_batch, _ in train_loader:
            q_batch = q_batch.to(device)
            r_batch = r_batch.to(device)

            optimizer.zero_grad()

            loss = saint_loss(
                best_model,
                q_batch,
                r_batch,
                criterion,
            )

            if loss is None:
                continue

            loss.backward()
            optimizer.step()

        # The epoch count is selected only from inner validation folds.
        # The outer test fold is not inspected during training.

    # --------------------------------------------------------
    # FINAL OUTER-FOLD EVALUATION AND OOF SAVING
    # --------------------------------------------------------

    fold_preds, fold_labels, fold_row_ids = (
        collect_saint_predictions(
            best_model,
            test_loader,
            include_row_ids=True,
        )
    )

    if not fold_labels:
        raise RuntimeError(
            f"No valid predictions were produced for outer fold {fold}."
        )

    if not (
        len(fold_preds)
        == len(fold_labels)
        == len(fold_row_ids)
    ):
        raise RuntimeError(
            f"Prediction, label, and row-ID lengths differ in fold {fold}."
        )

    if len(set(fold_row_ids)) != len(fold_row_ids):
        raise ValueError(
            f"Duplicate row_id values were produced in outer fold {fold}."
        )

    fold_metrics = calculate_metrics(
        fold_labels,
        fold_preds,
    )

    auc_per_fold.append(fold_metrics["auc"])
    acc_per_fold.append(fold_metrics["accuracy"])
    rmse_per_fold.append(fold_metrics["rmse"])
    mae_per_fold.append(fold_metrics["mae"])
    precision_per_fold.append(fold_metrics["precision"])
    recall_per_fold.append(fold_metrics["recall"])
    f1_per_fold.append(fold_metrics["f1"])

    all_preds_all_folds.extend(fold_preds)
    all_labels_all_folds.extend(fold_labels)
    all_rowids_all_folds.extend(fold_row_ids)
    all_foldnums_all_folds.extend(
        [fold] * len(fold_preds)
    )

    print(f"\nEvaluation on Fold {fold} Test Set:")
    print(f"AUC: {fold_metrics['auc']:.6f}")
    print(f"Accuracy: {fold_metrics['accuracy']:.6f}")
    print(f"RMSE: {fold_metrics['rmse']:.6f}")
    print(f"MAE: {fold_metrics['mae']:.6f}")
    print(f"Precision: {fold_metrics['precision']:.6f}")
    print(f"Recall: {fold_metrics['recall']:.6f}")
    print(f"F1 Score: {fold_metrics['f1']:.6f}")

    fold_df = pd.DataFrame({
        "row_id": fold_row_ids,
        "student_id": [
            student_id_lookup[row_id]
            for row_id in fold_row_ids
        ],
        "fold": fold,
        "y_true": fold_labels,
        "y_pred": fold_preds,
        "model_name": model_name,
        "kc_name": kc_name,
    })

    fold_outfile = os.path.join(
        oof_output_dir,
        f"oof_{model_name}_{kc_name}_fold{fold}.csv",
    )

    fold_df.to_csv(
        fold_outfile,
        index=False,
    )

    print(
        f"Saved Fold {fold} OOF predictions to "
        f"{fold_outfile}"
    )


# ============================================================
# SAVE COMBINED OOF FILE
# ============================================================

oof_df = pd.DataFrame({
    "row_id": all_rowids_all_folds,
    "student_id": [
        student_id_lookup[row_id]
        for row_id in all_rowids_all_folds
    ],
    "fold": all_foldnums_all_folds,
    "y_true": all_labels_all_folds,
    "y_pred": all_preds_all_folds,
    "model_name": model_name,
    "kc_name": kc_name,
}).sort_values(
    "row_id"
).reset_index(
    drop=True
)

if oof_df["row_id"].duplicated().any():
    duplicate_count = int(
        oof_df["row_id"].duplicated().sum()
    )

    raise ValueError(
        f"Combined OOF data contains "
        f"{duplicate_count} duplicate row_id values."
    )

found_oof_folds = sorted(
    oof_df["fold"].unique().tolist()
)

expected_oof_folds = list(
    range(1, n_splits + 1)
)

if found_oof_folds != expected_oof_folds:
    raise ValueError(
        f"Combined OOF file contains folds {found_oof_folds}; "
        f"expected {expected_oof_folds}."
    )

oof_outfile = os.path.join(
    oof_output_dir,
    f"oof_{model_name}_{kc_name}_all.csv",
)

oof_df.to_csv(
    oof_outfile,
    index=False,
)

print(
    f"\nSaved combined OOF predictions to "
    f"{oof_outfile}"
)


# ============================================================
# POOLED AND FOLD-AVERAGED RESULTS
# ============================================================

pooled_metrics = calculate_metrics(
    all_labels_all_folds,
    all_preds_all_folds,
)

avg_auc, std_auc = mean_std(auc_per_fold)
avg_acc, std_acc = mean_std(acc_per_fold)
avg_rmse, std_rmse = mean_std(rmse_per_fold)
avg_mae, std_mae = mean_std(mae_per_fold)
avg_precision, std_precision = mean_std(
    precision_per_fold
)
avg_recall, std_recall = mean_std(
    recall_per_fold
)
avg_f1, std_f1 = mean_std(f1_per_fold)

print("\n=== Final Evaluation Across All Folds ===")
print("Metric       | Pooled Score | Average Score ± Std")
print("-------------|--------------|---------------------")
print(
    f"AUC          | {pooled_metrics['auc']:.6f}      | "
    f"{avg_auc:.6f} ± {std_auc:.6f}"
)
print(
    f"Accuracy     | {pooled_metrics['accuracy']:.6f}      | "
    f"{avg_acc:.6f} ± {std_acc:.6f}"
)
print(
    f"RMSE         | {pooled_metrics['rmse']:.6f}      | "
    f"{avg_rmse:.6f} ± {std_rmse:.6f}"
)
print(
    f"MAE          | {pooled_metrics['mae']:.6f}      | "
    f"{avg_mae:.6f} ± {std_mae:.6f}"
)
print(
    f"Precision    | {pooled_metrics['precision']:.6f}      | "
    f"{avg_precision:.6f} ± {std_precision:.6f}"
)
print(
    f"Recall       | {pooled_metrics['recall']:.6f}      | "
    f"{avg_recall:.6f} ± {std_recall:.6f}"
)
print(
    f"F1 Score     | {pooled_metrics['f1']:.6f}      | "
    f"{avg_f1:.6f} ± {std_f1:.6f}"
)

print("\n=== OOF Summary ===")
print(f"Combined OOF rows: {len(oof_df)}")
print(f"Unique OOF row_ids: {oof_df['row_id'].nunique()}")
print(f"Combined OOF file: {oof_outfile}")

## After-feedback prediction


In [ ]:
# SAINT After Feedback

import os
import random
import itertools
from statistics import mean, stdev

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==== IMPORT SAINT FROM PYKT ====
from pykt.models import saint

SAINT = saint.SAINT


# ============================================================
# REPRODUCIBILITY AND DEVICE
# ============================================================

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")


# ============================================================
# SETTINGS
# ============================================================

seq_len = 20
batch_size = 8
n_splits = 3
max_epochs = 100
patience = 10

model_name = "SAINT"
kc_name = "actionableelementid"
question_col = "KC (actionableelementid)"
correct_col = "eventualcorrect"
student_col = "Anon Student Id"

fixed_fold_file = "data/fixed_outer_folds_af.csv"
oof_output_dir = "SAINT_AF"

os.makedirs(oof_output_dir, exist_ok=True)


# ============================================================
# LOAD DATA
# ============================================================

# Load your dataframe before running this script, for example:
# logging_data = pd.read_csv("your_file.csv")

if "logging_data" not in globals():
    raise NameError(
        "logging_data is not defined. Load your dataframe into a variable "
        "named logging_data before running this script."
    )

logging_data = logging_data.copy().reset_index(drop=True)

# row_id must be created on the full original dataframe before model-specific
# filtering so OOF predictions can align across all DLKT models.
if "row_id" not in logging_data.columns:
    logging_data["row_id"] = np.arange(len(logging_data), dtype=np.int64)

if logging_data["row_id"].duplicated().any():
    raise ValueError("row_id must be unique in the full logging_data dataframe.")

required_source_columns = {
    "row_id",
    question_col,
    correct_col,
    student_col,
}

missing_source_columns = required_source_columns - set(logging_data.columns)

if missing_source_columns:
    raise ValueError(
        f"logging_data is missing required columns: "
        f"{sorted(missing_source_columns)}"
    )


# ============================================================
# FIXED OUTER FOLDS
# ============================================================

# Remove any old fold columns left by notebook reruns before merging.
old_fold_columns = [
    column
    for column in logging_data.columns
    if column == "outer_fold" or column.startswith("outer_fold_")
]

if old_fold_columns:
    logging_data = logging_data.drop(columns=old_fold_columns)

if os.path.exists(fixed_fold_file):
    fixed_folds = pd.read_csv(fixed_fold_file)

    required_fold_columns = {"row_id", "outer_fold"}
    missing_fold_columns = required_fold_columns - set(fixed_folds.columns)

    if missing_fold_columns:
        raise ValueError(
            f"{fixed_fold_file} is missing columns: "
            f"{sorted(missing_fold_columns)}"
        )

    if fixed_folds["row_id"].duplicated().any():
        raise ValueError(
            f"{fixed_fold_file} contains duplicate row_id values."
        )

    found_folds = sorted(
        fixed_folds["outer_fold"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    expected_folds = list(range(1, n_splits + 1))

    if found_folds != expected_folds:
        raise ValueError(
            f"{fixed_fold_file} contains folds {found_folds}, "
            f"but this run expects {expected_folds}."
        )

    logging_data = logging_data.merge(
        fixed_folds[["row_id", "outer_fold"]],
        on="row_id",
        how="left",
        validate="one_to_one",
    )

    missing_valid_fold = (
        logging_data[student_col].notna()
        & logging_data["outer_fold"].isna()
    )

    if missing_valid_fold.any():
        raise ValueError(
            "Some rows with valid student IDs are missing outer-fold "
            "assignments. The fixed-fold file may belong to another "
            "dataset or row ordering."
        )

    print(f"Loaded fixed folds from {fixed_fold_file}")

else:
    base_for_folds = logging_data.dropna(
        subset=[student_col]
    ).copy()

    if base_for_folds[student_col].nunique() < n_splits:
        raise ValueError(
            f"At least {n_splits} unique students are required to create "
            f"{n_splits} grouped outer folds."
        )

    base_for_folds["outer_fold"] = -1

    outer_gkf = GroupKFold(n_splits=n_splits)

    for fold_idx, (_, test_idx) in enumerate(
        outer_gkf.split(
            base_for_folds,
            groups=base_for_folds[student_col],
        ),
        start=1,
    ):
        base_for_folds.iloc[
            test_idx,
            base_for_folds.columns.get_loc("outer_fold"),
        ] = fold_idx

    fixed_folds = base_for_folds[
        ["row_id", "outer_fold"]
    ].copy()

    fixed_folds.to_csv(
        fixed_fold_file,
        index=False,
    )

    logging_data = logging_data.merge(
        fixed_folds,
        on="row_id",
        how="left",
        validate="one_to_one",
    )

    print(f"Saved fixed folds to {fixed_fold_file}")

logging_data["outer_fold"] = logging_data["outer_fold"].astype("Int64")


# ============================================================
# MODEL-SPECIFIC PREPROCESSING
# ============================================================

logging_model = logging_data.dropna(
    subset=[
        question_col,
        correct_col,
        student_col,
        "outer_fold",
    ]
).copy()

logging_model[correct_col] = logging_model[correct_col].astype(int)
logging_model["outer_fold"] = logging_model["outer_fold"].astype(int)

invalid_labels = ~logging_model[correct_col].isin([0, 1])

if invalid_labels.any():
    bad_values = sorted(
        logging_model.loc[invalid_labels, correct_col]
        .unique()
        .tolist()
    )

    raise ValueError(
        f"{correct_col} must be binary 0/1. Found: {bad_values}"
    )

# Map each original row to its student so student_id is saved with OOF rows.
student_id_lookup = (
    logging_model[["row_id", student_col]]
    .drop_duplicates(subset=["row_id"])
    .set_index("row_id")[student_col]
    .to_dict()
)

print(f"seq_len: {seq_len}")
print(f"Usable rows: {len(logging_model)}")
print(f"Unique students: {logging_model[student_col].nunique()}")

# Global KC mapping for this model run.
# 0 is reserved exclusively for padding.
all_qids = logging_model[question_col].dropna().unique()

qid_to_index = {
    qid: index + 1
    for index, qid in enumerate(all_qids)
}

num_questions = len(qid_to_index) + 1

print(f"Number of KC IDs including padding: {num_questions}")


# ============================================================
# DATASET
# ============================================================

class KTDataFromLogging(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        seq_len: int,
        question_col: str,
        correct_col: str,
        student_col: str,
        qid_to_index: dict,
    ) -> None:
        self.seq_len = seq_len
        self.samples = []

        df = df.copy()

        df["qid_index"] = (
            df[question_col]
            .map(qid_to_index)
            .fillna(0)
            .astype(int)
        )

        # Preserve the existing dataframe order within each student,
        # exactly like the SKVMN wrapper.
        for _, group in df.groupby(student_col, sort=False):
            q_seq = group["qid_index"].tolist()
            r_seq = group[correct_col].astype(int).tolist()
            row_seq = group["row_id"].astype(int).tolist()

            for start in range(0, len(q_seq), seq_len - 1):
                end = min(start + seq_len, len(q_seq))

                if end - start < 2:
                    break

                q_chunk = q_seq[start:end]
                r_chunk = r_seq[start:end]
                row_chunk = row_seq[start:end]

                pad_len = seq_len - len(q_chunk)

                if pad_len > 0:
                    q_chunk += [0] * pad_len
                    r_chunk += [0] * pad_len
                    row_chunk += [-1] * pad_len

                self.samples.append(
                    (
                        torch.tensor(q_chunk, dtype=torch.long),
                        torch.tensor(r_chunk, dtype=torch.long),
                        torch.tensor(row_chunk, dtype=torch.long),
                    )
                )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        return self.samples[index]


# ============================================================
# SAINT HYPERPARAMETER GRID
# ============================================================

# Same exhaustive search space as the original SAINT wrapper,
# searched with ordinary Python exhaustive loops.
param_search_space = {
    "d_model": [64, 128, 256],
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "num_attn_heads": [4, 8],
    "dropout": [0.1, 0.3, 0.5],
    "n_blocks": [1, 2, 4],
}


# ============================================================
# HELPERS
# ============================================================

def rmse_score(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return float(
        np.sqrt(
            np.mean((y_true - y_pred) ** 2)
        )
    )


def make_saint_model(params: dict) -> nn.Module:
    """
    This experiment varies ONE KC representation at a time.

    SAINT normally supports a question/exercise stream (num_q) and a
    concept/category stream (num_c). Here the selected KC column is used
    only as the concept/category stream so it is not embedded twice.
    """
    return SAINT(
        num_q=0,
        num_c=num_questions,
        seq_len=seq_len,
        emb_size=params["d_model"],
        num_attn_heads=params["num_attn_heads"],
        dropout=params["dropout"],
        n_blocks=params["n_blocks"],
        emb_type="qid",
    ).to(device)


def saint_forward(
    model: nn.Module,
    q_batch: torch.Tensor,
    r_batch: torch.Tensor,
) -> torch.Tensor:
    """
    pyKT SAINT alignment for our full-sequence wrapper.

    Encoder:
        full KC sequence, length L

    Decoder:
        responses r[:, :-1], length L-1
        SAINT prepends its own start token -> decoder length L

    Therefore:
        output[:, 1:] predicts r[:, 1:]
    """
    dummy_exercise = torch.zeros_like(q_batch)

    outputs = model(
        in_ex=dummy_exercise,
        in_cat=q_batch,
        in_res=r_batch[:, :-1],
    )

    if outputs.shape != q_batch.shape:
        raise RuntimeError(
            f"Unexpected SAINT output shape {tuple(outputs.shape)}; "
            f"expected {tuple(q_batch.shape)}."
        )

    return outputs


def saint_loss(
    model: nn.Module,
    q_batch: torch.Tensor,
    r_batch: torch.Tensor,
    criterion: nn.Module,
):
    """
    SAINT-specific training alignment.

    output[:, 1:] -> r_batch[:, 1:]
    Position 0 and padding are excluded.
    """
    outputs = saint_forward(
        model,
        q_batch,
        r_batch,
    )

    valid_mask = q_batch[:, 1:] != 0

    if not valid_mask.any():
        return None

    loss = criterion(
        outputs[:, 1:][valid_mask],
        r_batch[:, 1:].float()[valid_mask],
    )

    return loss


def collect_saint_predictions(
    model: nn.Module,
    loader: DataLoader,
    include_row_ids: bool = False,
):
    """
    SAINT evaluation/OOF alignment:

        outputs[b, t] -> r_batch[b, t] -> rowid_batch[b, t]

    Position 0 is excluded.
    Padding is excluded because KC ID 0 is reserved for padding.
    """
    model.eval()

    preds_all = []
    labels_all = []
    row_ids_all = []

    with torch.no_grad():
        for q_batch, r_batch, rowid_batch in loader:
            q_batch = q_batch.to(device)
            r_batch = r_batch.to(device)

            outputs = saint_forward(
                model,
                q_batch,
                r_batch,
            )

            for batch_index in range(q_batch.size(0)):
                for time_index in range(1, seq_len):
                    if q_batch[batch_index, time_index].item() == 0:
                        continue

                    pred = outputs[
                        batch_index,
                        time_index,
                    ].item()

                    label = r_batch[
                        batch_index,
                        time_index,
                    ].item()

                    if include_row_ids:
                        row_id = rowid_batch[
                            batch_index,
                            time_index,
                        ].item()

                        if row_id == -1:
                            continue

                        row_ids_all.append(int(row_id))

                    preds_all.append(float(pred))
                    labels_all.append(int(label))

    if include_row_ids:
        return preds_all, labels_all, row_ids_all

    return preds_all, labels_all


def calculate_metrics(labels, predictions) -> dict:
    labels = np.asarray(labels, dtype=int)
    predictions = np.asarray(predictions, dtype=float)
    binary_predictions = (predictions > 0.5).astype(int)

    auc = (
        roc_auc_score(labels, predictions)
        if len(np.unique(labels)) > 1
        else np.nan
    )

    return {
        "auc": auc,
        "accuracy": accuracy_score(labels, binary_predictions),
        "rmse": rmse_score(labels, predictions),
        "mae": mean_absolute_error(labels, predictions),
        "precision": precision_score(
            labels,
            binary_predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            labels,
            binary_predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            labels,
            binary_predictions,
            zero_division=0,
        ),
    }


def mean_std(values):
    valid_values = [
        float(value)
        for value in values
        if not pd.isna(value)
    ]

    if not valid_values:
        return np.nan, np.nan

    return (
        mean(valid_values),
        stdev(valid_values)
        if len(valid_values) > 1
        else 0.0,
    )


def parameter_combinations():
    keys = list(param_search_space.keys())

    for values in itertools.product(
        *(param_search_space[key] for key in keys)
    ):
        yield dict(zip(keys, values))


# ============================================================
# STORAGE
# ============================================================

all_preds_all_folds = []
all_labels_all_folds = []
all_rowids_all_folds = []
all_foldnums_all_folds = []

auc_per_fold = []
acc_per_fold = []
rmse_per_fold = []
mae_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []


# ============================================================
# OUTER CROSS-VALIDATION
# ============================================================

for fold in range(1, n_splits + 1):
    print(f"\n=== Outer Fold {fold}/{n_splits} ===")

    train_val_df = logging_model[
        logging_model["outer_fold"] != fold
    ].copy()

    test_df = logging_model[
        logging_model["outer_fold"] == fold
    ].copy()

    print(f"Train/validation rows: {len(train_val_df)}")
    print(f"Test rows: {len(test_df)}")
    print(
        f"Train/validation students: "
        f"{train_val_df[student_col].nunique()}"
    )
    print(f"Test students: {test_df[student_col].nunique()}")

    if train_val_df[student_col].nunique() < n_splits:
        raise ValueError(
            f"Outer fold {fold} does not contain enough training students "
            f"for {n_splits}-fold inner GroupKFold."
        )

    inner_cv = GroupKFold(n_splits=n_splits)
    inner_groups = train_val_df[student_col]

    # --------------------------------------------------------
    # INNER-CV HYPERPARAMETER SEARCH
    # --------------------------------------------------------

    best_params = None
    best_mean_inner_auc = -np.inf
    recommended_epochs = 1

    combinations = list(parameter_combinations())

    # Match the same compact trial-progress display used by the other runs.
    progress_bar = tqdm(total=len(combinations))

    best_trial_index = None

    for combination_index, params in enumerate(combinations, start=1):
        auc_scores = []
        best_epochs = []

        for inner_train_idx, inner_val_idx in inner_cv.split(
            train_val_df,
            groups=inner_groups,
        ):
            inner_train_df = train_val_df.iloc[
                inner_train_idx
            ].copy()

            inner_val_df = train_val_df.iloc[
                inner_val_idx
            ].copy()

            train_dataset = KTDataFromLogging(
                inner_train_df,
                seq_len,
                question_col,
                correct_col,
                student_col,
                qid_to_index,
            )

            val_dataset = KTDataFromLogging(
                inner_val_df,
                seq_len,
                question_col,
                correct_col,
                student_col,
                qid_to_index,
            )

            train_loader = DataLoader(
                train_dataset,
                batch_size=batch_size,
                shuffle=True,
            )

            val_loader = DataLoader(
                val_dataset,
                batch_size=batch_size,
                shuffle=False,
            )

            model = make_saint_model(params)
            criterion = nn.BCELoss()

            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=params["learning_rate"],
            )

            best_auc_inner = -np.inf
            best_epoch_inner = 1
            no_improve = 0

            for epoch in range(max_epochs):
                model.train()

                for q_batch, r_batch, _ in train_loader:
                    q_batch = q_batch.to(device)
                    r_batch = r_batch.to(device)

                    optimizer.zero_grad()

                    loss = saint_loss(
                        model,
                        q_batch,
                        r_batch,
                        criterion,
                    )

                    if loss is None:
                        continue

                    loss.backward()
                    optimizer.step()

                val_preds, val_labels = collect_saint_predictions(
                    model,
                    val_loader,
                    include_row_ids=False,
                )

                if (
                    len(val_labels) > 0
                    and len(np.unique(val_labels)) > 1
                ):
                    val_auc = roc_auc_score(
                        val_labels,
                        val_preds,
                    )

                    # Same rule as the SKVMN wrapper:
                    # any strictly larger AUC counts as improvement.
                    if val_auc > best_auc_inner:
                        best_auc_inner = val_auc
                        best_epoch_inner = epoch + 1
                        no_improve = 0
                    else:
                        no_improve += 1

                    if no_improve >= patience:
                        break

            auc_scores.append(best_auc_inner)
            best_epochs.append(best_epoch_inner)

        mean_inner_auc = (
            float(np.mean(auc_scores))
            if auc_scores
            else 0.0
        )

        print(
            f"Mean inner AUC for {params}: "
            f"{mean_inner_auc:.6f}"
        )

        if mean_inner_auc > best_mean_inner_auc:
            best_mean_inner_auc = mean_inner_auc
            best_params = params.copy()
            recommended_epochs = max(
                1, int(np.median(best_epochs))
            )
            # Trial numbering starts at 0 to match the other model runs.
            best_trial_index = combination_index - 1

        # Update the description first, then increment the completed-trial count.
        if best_trial_index is not None:
            progress_bar.set_description(
                f"Best trial: {best_trial_index}. "
                f"Best value: {best_mean_inner_auc:.6g}"
            )

        progress_bar.update(1)

    progress_bar.close()

    if best_params is None:
        raise RuntimeError(
            f"No valid SAINT parameter configuration was selected "
            f"for outer fold {fold}."
        )

    print(
        f"\nBest params for fold {fold}: "
        f"{best_params}, AUC: {best_mean_inner_auc:.6f}"
    )


    # --------------------------------------------------------
    # RETRAIN BEST CONFIGURATION ON TRAIN_VAL_DF
    # --------------------------------------------------------

    train_dataset = KTDataFromLogging(
        train_val_df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index,
    )

    test_dataset = KTDataFromLogging(
        test_df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    best_model = make_saint_model(best_params)
    criterion = nn.BCELoss()

    optimizer = torch.optim.Adam(
        best_model.parameters(),
        lr=best_params["learning_rate"],
    )

    # Retrain on every outer-training student for the epoch count
    # selected from the inner validation folds.
    for epoch in range(recommended_epochs):
        best_model.train()

        for q_batch, r_batch, _ in train_loader:
            q_batch = q_batch.to(device)
            r_batch = r_batch.to(device)

            optimizer.zero_grad()

            loss = saint_loss(
                best_model,
                q_batch,
                r_batch,
                criterion,
            )

            if loss is None:
                continue

            loss.backward()
            optimizer.step()

        # The epoch count is selected only from inner validation folds.
        # The outer test fold is not inspected during training.

    # --------------------------------------------------------
    # FINAL OUTER-FOLD EVALUATION AND OOF SAVING
    # --------------------------------------------------------

    fold_preds, fold_labels, fold_row_ids = (
        collect_saint_predictions(
            best_model,
            test_loader,
            include_row_ids=True,
        )
    )

    if not fold_labels:
        raise RuntimeError(
            f"No valid predictions were produced for outer fold {fold}."
        )

    if not (
        len(fold_preds)
        == len(fold_labels)
        == len(fold_row_ids)
    ):
        raise RuntimeError(
            f"Prediction, label, and row-ID lengths differ in fold {fold}."
        )

    if len(set(fold_row_ids)) != len(fold_row_ids):
        raise ValueError(
            f"Duplicate row_id values were produced in outer fold {fold}."
        )

    fold_metrics = calculate_metrics(
        fold_labels,
        fold_preds,
    )

    auc_per_fold.append(fold_metrics["auc"])
    acc_per_fold.append(fold_metrics["accuracy"])
    rmse_per_fold.append(fold_metrics["rmse"])
    mae_per_fold.append(fold_metrics["mae"])
    precision_per_fold.append(fold_metrics["precision"])
    recall_per_fold.append(fold_metrics["recall"])
    f1_per_fold.append(fold_metrics["f1"])

    all_preds_all_folds.extend(fold_preds)
    all_labels_all_folds.extend(fold_labels)
    all_rowids_all_folds.extend(fold_row_ids)
    all_foldnums_all_folds.extend(
        [fold] * len(fold_preds)
    )

    print(f"\nEvaluation on Fold {fold} Test Set:")
    print(f"AUC: {fold_metrics['auc']:.6f}")
    print(f"Accuracy: {fold_metrics['accuracy']:.6f}")
    print(f"RMSE: {fold_metrics['rmse']:.6f}")
    print(f"MAE: {fold_metrics['mae']:.6f}")
    print(f"Precision: {fold_metrics['precision']:.6f}")
    print(f"Recall: {fold_metrics['recall']:.6f}")
    print(f"F1 Score: {fold_metrics['f1']:.6f}")

    fold_df = pd.DataFrame({
        "row_id": fold_row_ids,
        "student_id": [
            student_id_lookup[row_id]
            for row_id in fold_row_ids
        ],
        "fold": fold,
        "y_true": fold_labels,
        "y_pred": fold_preds,
        "model_name": model_name,
        "kc_name": kc_name,
    })

    fold_outfile = os.path.join(
        oof_output_dir,
        f"oof_{model_name}_{kc_name}_fold{fold}.csv",
    )

    fold_df.to_csv(
        fold_outfile,
        index=False,
    )

    print(
        f"Saved Fold {fold} OOF predictions to "
        f"{fold_outfile}"
    )


# ============================================================
# SAVE COMBINED OOF FILE
# ============================================================

oof_df = pd.DataFrame({
    "row_id": all_rowids_all_folds,
    "student_id": [
        student_id_lookup[row_id]
        for row_id in all_rowids_all_folds
    ],
    "fold": all_foldnums_all_folds,
    "y_true": all_labels_all_folds,
    "y_pred": all_preds_all_folds,
    "model_name": model_name,
    "kc_name": kc_name,
}).sort_values(
    "row_id"
).reset_index(
    drop=True
)

if oof_df["row_id"].duplicated().any():
    duplicate_count = int(
        oof_df["row_id"].duplicated().sum()
    )

    raise ValueError(
        f"Combined OOF data contains "
        f"{duplicate_count} duplicate row_id values."
    )

found_oof_folds = sorted(
    oof_df["fold"].unique().tolist()
)

expected_oof_folds = list(
    range(1, n_splits + 1)
)

if found_oof_folds != expected_oof_folds:
    raise ValueError(
        f"Combined OOF file contains folds {found_oof_folds}; "
        f"expected {expected_oof_folds}."
    )

oof_outfile = os.path.join(
    oof_output_dir,
    f"oof_{model_name}_{kc_name}_all.csv",
)

oof_df.to_csv(
    oof_outfile,
    index=False,
)

print(
    f"\nSaved combined OOF predictions to "
    f"{oof_outfile}"
)


# ============================================================
# POOLED AND FOLD-AVERAGED RESULTS
# ============================================================

pooled_metrics = calculate_metrics(
    all_labels_all_folds,
    all_preds_all_folds,
)

avg_auc, std_auc = mean_std(auc_per_fold)
avg_acc, std_acc = mean_std(acc_per_fold)
avg_rmse, std_rmse = mean_std(rmse_per_fold)
avg_mae, std_mae = mean_std(mae_per_fold)
avg_precision, std_precision = mean_std(
    precision_per_fold
)
avg_recall, std_recall = mean_std(
    recall_per_fold
)
avg_f1, std_f1 = mean_std(f1_per_fold)

print("\n=== Final Evaluation Across All Folds ===")
print("Metric       | Pooled Score | Average Score ± Std")
print("-------------|--------------|---------------------")
print(
    f"AUC          | {pooled_metrics['auc']:.6f}      | "
    f"{avg_auc:.6f} ± {std_auc:.6f}"
)
print(
    f"Accuracy     | {pooled_metrics['accuracy']:.6f}      | "
    f"{avg_acc:.6f} ± {std_acc:.6f}"
)
print(
    f"RMSE         | {pooled_metrics['rmse']:.6f}      | "
    f"{avg_rmse:.6f} ± {std_rmse:.6f}"
)
print(
    f"MAE          | {pooled_metrics['mae']:.6f}      | "
    f"{avg_mae:.6f} ± {std_mae:.6f}"
)
print(
    f"Precision    | {pooled_metrics['precision']:.6f}      | "
    f"{avg_precision:.6f} ± {std_precision:.6f}"
)
print(
    f"Recall       | {pooled_metrics['recall']:.6f}      | "
    f"{avg_recall:.6f} ± {std_recall:.6f}"
)
print(
    f"F1 Score     | {pooled_metrics['f1']:.6f}      | "
    f"{avg_f1:.6f} ± {std_f1:.6f}"
)

print("\n=== OOF Summary ===")
print(f"Combined OOF rows: {len(oof_df)}")
print(f"Unique OOF row_ids: {oof_df['row_id'].nunique()}")
print(f"Combined OOF file: {oof_outfile}")